In [6]:
TOKEN = "REDACTED_ENTSOE_KEY"

from entsoe import EntsoePandasClient
import pandas as pd

client = EntsoePandasClient(api_key=TOKEN)
start = pd.Timestamp('20260401', tz='Europe/Paris')
end = pd.Timestamp('20260514', tz='Europe/Paris')

# Prix spot France
prices = client.query_day_ahead_prices('FR', start=start, end=end)
prices = prices.reset_index().rename(columns={"index":"time","0":"price"})
# Bonus très utile pour ton projet : 
#generation = client.query_generation('FR', start=start, end=end).reset_index().rename(columns={"index":"time"})  # mix réel
load = client.query_load('FR', start=start, end=end).reset_index().rename(columns={"index":"time"})  # demande
#wind_solar_forecast = client.query_wind_and_solar_forecast('FR', start=start, end=end).reset_index().rename(columns={"index":"time"})

In [5]:
prices["time"] = pd.to_datetime(prices["time"])

In [9]:
start_ts = start
end_ts = end + pd.Timedelta(days=1)


load_fc = client.query_load_forecast("FR", start=start_ts, end=end_ts)
load_fc

,Forecasted Load
2026-04-01 00:00:00+02:00,56000.0
2026-04-01 00:15:00+02:00,55200.0
2026-04-01 00:30:00+02:00,54400.0
2026-04-01 00:45:00+02:00,53700.0
2026-04-01 01:00:00+02:00,53000.0
...,...
2026-05-14 22:45:00+02:00,46500.0
2026-05-14 23:00:00+02:00,47100.0
2026-05-14 23:15:00+02:00,47050.0
2026-05-14 23:30:00+02:00,47000.0


In [15]:
geo_path = "/Users/maximeanno/Documents/ProjetPerso/EnergeticPricing/data/Geographie/departements.geojson"
print(f"[load_data] Lecture du geojson : {geo_path}")
import geopandas as gpd 
departements = gpd.read_file(geo_path)

departement = departements[departements["code"] == "22"]
point = departement.geometry.representative_point().iloc[0]
latitude, longitude = point.y, point.x
latitude = 48.457
longitude = 2.750

[load_data] Lecture du geojson : /Users/maximeanno/Documents/ProjetPerso/EnergeticPricing/data/Geographie/departements.geojson


In [16]:
import requests
import pandas as pd

# Paris - historique horaire
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": "2012-01-01",
    "end_date": "2026-05-14",
    "hourly": "temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,precipitation",
    "timezone": "Europe/Paris"
}
r = requests.get(url, params=params)
data = r.json()
df = pd.DataFrame(data["hourly"])
df["time"] = pd.to_datetime(df["time"])

In [18]:
df

,time,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,precipitation
0,2012-01-01 00:00:00,11.7,21.6,0.0,100,0.1
1,2012-01-01 01:00:00,11.8,22.6,0.0,100,0.2
2,2012-01-01 02:00:00,11.8,19.5,0.0,100,0.0
3,2012-01-01 03:00:00,11.7,17.4,0.0,100,0.0
4,2012-01-01 04:00:00,11.5,18.1,0.0,100,0.0
...,...,...,...,...,...,...
125947,2026-05-14 19:00:00,9.7,7.2,289.0,44,1.0
125948,2026-05-14 20:00:00,9.0,8.4,179.0,30,0.6
125949,2026-05-14 21:00:00,8.1,4.4,67.0,11,0.0
125950,2026-05-14 22:00:00,6.8,5.2,2.0,14,0.0


In [7]:
import requests
import pandas as pd

# Paris - historique horaire
url = "https://api.open-meteo.com/v1/forecast"
# mêmes params, avec forecast_days=2 par exemple

params = {
    "latitude": 48.85,
    "longitude": 2.35,
    "forecast_days": 2,
    "hourly": "temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,precipitation",
    "timezone": "Europe/Paris"
}
r = requests.get(url, params=params)
data = r.json()
df = pd.DataFrame(data["hourly"])
df["time"] = pd.to_datetime(df["time"])

In [8]:
df

,time,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,precipitation
0,2026-04-18 00:00:00,16.3,3.1,0.0,5,0.0
1,2026-04-18 01:00:00,15.4,3.6,0.0,11,0.0
2,2026-04-18 02:00:00,14.2,4.8,0.0,43,0.0
3,2026-04-18 03:00:00,13.7,3.8,0.0,88,0.0
4,2026-04-18 04:00:00,13.4,4.7,0.0,97,0.0
5,2026-04-18 05:00:00,12.7,4.7,0.0,97,0.0
6,2026-04-18 06:00:00,12.8,2.5,0.0,100,0.0
7,2026-04-18 07:00:00,12.8,1.8,0.0,64,0.0
8,2026-04-18 08:00:00,12.8,4.6,29.8,82,0.0
9,2026-04-18 09:00:00,13.7,4.6,144.8,90,0.0
